In [ ]:
import matplotlib.pyplot as plt
import copy
from astropy.time import Time
import numpy as np

from orbitalsim import diagnostic_calculators
from orbitalsim.universal_constants import UniversalConstants
from orbitalsim.celestial_body import CelestialBody
from orbitalsim.planets import Planets
from orbitalsim.diagnostic_calculators import *
from orbitalsim.simulation import NBodySimulation
from orbitalsim.set_positions import PositionsSetter

from orbitalsim.integrators.symplectic_euler import SymplecticEulerIntegrator
from orbitalsim.integrators.yoshida_4th_order import Yoshida4thOrderIntegrator
from orbitalsim.integrators.forward_euler import ForwardEuler
from orbitalsim.integrators.runge_kutta_4 import RungeKutta4Integrator
from orbitalsim.integrators.velocity_verlet import VelocityVerletIntegrator

In [ ]:
class SimulationRunner:

    integrator = Yoshida4thOrderIntegrator()

    sun = CelestialBody(1.989e30, np.array([0.0, 0.0, 0.0]), np.array([0.0, 0.0, 0.0]), name="Sun")

    all_bodies = copy.deepcopy(Planets().allPlanets)
    all_bodies.append(sun)

    center_of_mass = sum(body.mass * body.position for body in all_bodies) / sum(body.mass for body in all_bodies)

    for body in all_bodies:
        body.position = body.position - center_of_mass

    total_momentum = sum(body.mass * body.velocity for body in all_bodies)

    for body in all_bodies:
        if body.name == "Sun":
            body.velocity = body.velocity - total_momentum / body.mass

    simulation = NBodySimulation(integrator, bodies=all_bodies)

    traj, _, _, _, _, _, _, _ = simulation.simulator(2000, 50000)

    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection="3d")

    for i, body in enumerate(simulation.bodies):
        ax.plot(
            traj[:, i, 0],
            traj[:, i, 1],
            traj[:, i, 2],
            label = body.name
        )

        # final pos
        ax.scatter(
            traj[-1, i, 0],
            traj[-1, i, 1],
            traj[-1, i, 2],
        )

    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")
    ax.legend()

    plt.show()


In [18]:
class SimulationDebugRunner:

    sun = CelestialBody(1.32712440018e20 / UniversalConstants.G, np.array([0.0, 0.0, 0.0]), np.array([0.0, 0.0, 0.0]), name="Sun", horizons_id='10')

    all_bodies = copy.deepcopy(Planets().allPlanets)
    all_bodies.append(sun)

    center_of_mass = sum(body.mass * body.position for body in all_bodies) / sum(body.mass for body in all_bodies)

    for body in all_bodies:
        body.position = body.position - center_of_mass

    total_momentum = sum(body.mass * body.velocity for body in all_bodies)

    for body in all_bodies:
        if body.name == "Sun":
            body.velocity = body.velocity - total_momentum / body.mass

    start_time = "2000-01-01 00:00:00"
    epoch = Time(start_time).tdb.jd
    PositionsSetter.set_positions_at_time(all_bodies,epoch)

    #print(diagnostic_calculators.two_simulation_position_comparison(all_bodies, 200000, Yoshida4thOrderIntegrator(), RungeKutta4Integrator(), 10000, 100))

    print(diagnostic_calculators.horizon_data_position_comparison(all_bodies, start_time, 20000, RungeKutta4Integrator(), 10000))

[735817.74277171 566221.77440745 388584.55246018 204992.82907246
  35010.7418203    4921.26339402   1272.8561505    2342.00219402
   3246.93981217]


In [ ]:
import urllib.request

url = "https://naif.jpl.nasa.gov/pub/naif/generic_kernels/pck/pck00010.tpc"
urllib.request.urlretrieve(url, "../pck00010.tpc")
print("Downloaded successfully!")

url = "https://naif.jpl.nasa.gov/pub/naif/generic_kernels/pck/gm_de431.tpc"
urllib.request.urlretrieve(url, "../gm_de431.tpc")
print("Downloaded successfully!")